# Aggregate annual data

**1. Peak SWE:** the maximum daily SWE (ensemble median) in each water year, per pixel, for each basin.

**2. Yearly storm and climate data**, per basin and water year (1980–2025, the complete years in GridMET/ERA5):
- `Count`: number of GridMET-period storms
- `Max_IVT`: highest storm IVT of the year [kg m⁻¹ s⁻¹]
- `Winter_Temp`: mean Dec–Mar GridMET temperature [K] in elevation bins 2–4 (1500–1800, 1800–2500, > 2500 m)

## Inputs
| File | From | Used for |
|------|------|----------|
| `Data/Cleaned/<basin>/<basin>_SWE_WY*.nc4` | SWE cleaning script | Peak SWE |
| `Data/Cleaned/<basin>_storms_long.nc4` | Storm temperature/IVT notebook | Storm counts and IVT |
| `Data/Cleaned/GridMET/temp_avg_da.nc4` | GridMET notebook | Winter temperature |
| `Data/DEM/coarsened.nc4` | DEM notebook | Elevation bins on the GridMET grid |
| `config.py` / `utils.py` | | `data_folder`, `Basins`, `basin_names_list`, `shapes`, `years` / `calculate_water_year` |

## Outputs (in `Data/Cleaned/`)
| File | Contents |
|------|----------|
| `<basin>_peak_swe.nc4` | `SWE_Post` (WY, Latitude, Longitude): peak SWE [m] |
| `yearly_data.nc4` | `Count`, `Max_IVT` (Basin, WY); `Winter_Temp` (Basin, ElevationBin, WY) |

In [ ]:
# %% Imports and settings
import os

import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr

from config import data_folder, Basins, basin_names_list, shapes, years_swe, elev_bins, water_years_prec
from utils import calculate_water_year

cleaned = os.path.join(data_folder, 'Data/Cleaned')
WINTER_MONTHS = [12, 1, 2, 3]
BINS = [2, 3, 4]                     # bin i covers elev_bins[i-1] to \elev_bins[i]

In [ ]:
# %% 1) Peak SWE per water year
for basin in Basins:
    peaks = []
    for year in years_swe:
        f = os.path.join(cleaned, basin, f'{basin}_SWE_WY{year}_{(year + 1) % 100:02d}.nc4')
        swe = xr.open_dataset(f, chunks='auto')['SWE_Post'].sel(Stats=2)  # ensemble median
        peaks.append(swe.max('Day').drop_vars('Stats').expand_dims(WY=[year + 1]))
    xr.concat(peaks, 'WY').to_netcdf(os.path.join(cleaned, f'{basin}_peak_swe.nc4'))

In [ ]:
# %% 2a) Winter GridMET temperature, with elevation on the same grid
temp = xr.open_dataarray(os.path.join(cleaned, 'GridMET/temp_avg_da.nc4'), chunks='auto')
temp = (temp.sel(Day=temp['Day'].dt.month.isin(WINTER_MONTHS))  # winter days only: ~1/3 the memory
            .load(scheduler='threads')
            .rio.write_crs('EPSG:4326')
            .rio.set_spatial_dims(x_dim='Longitude', y_dim='Latitude'))
dem = xr.open_dataset(os.path.join(data_folder, 'Data/DEM/coarsened.nc4'))['Elevation'].interp_like(temp)

In [ ]:
# %% 2b) Yearly data per basin
def yearly_storm_stats(basin):
    """Storm count and max storm IVT per water year (GridMET-period storms)."""
    storms = xr.open_dataset(os.path.join(cleaned, f'{basin}_storms_long.nc4')).set_coords('WY')
    by_wy = storms['IVT'].groupby('WY')
    return xr.Dataset({
        'Max_IVT': by_wy.max().reindex(WY=water_years_prec),
        'Count': by_wy.count().reindex(WY=water_years_prec).fillna(0),  # years with no storms -> 0
    })


def winter_temp_by_bin(basin_name):
    """Mean Dec–Mar temperature [K] per water year in each elevation bin of the basin."""
    polygon = shapes.loc[shapes['name'] == basin_name, 'geometry'].iloc[0]
    t = temp.rio.clip([polygon], crs='EPSG:4326')

    per_bin = []
    for b in BINS:
        in_bin = (dem >= elev_bins[b - 1]) & (dem < elev_bins[b])
        daily = t.where(in_bin).mean(['Latitude', 'Longitude'])
        per_bin.append(daily.assign_coords(WY=calculate_water_year(daily['Day'])).groupby('WY').mean())

    return xr.concat(per_bin, dim=pd.Index(BINS, name='ElevationBin')).reindex(WY=water_years_prec).rename('Winter_Temp')


yearly = xr.concat(
    [xr.merge([yearly_storm_stats(b), winter_temp_by_bin(n)]) for b, n in zip(Basins, basin_names_list)],
    dim=pd.Index(Basins, name='Basin'),
)
yearly.to_netcdf(os.path.join(cleaned, 'yearly_data.nc4'))
yearly